# 05 · Statistics, sensitivity analysis, confidence intervals, tables and figures

Every number, table and figure of the paper is computed here **from the single final run** in
`Results/rerun/` written by `04_rerun_with_validation.ipynb` (editor comment 7). Nothing is typed in by hand except the six published
MoleculeACE cliff percentages used for the comparison in Section 2.9.

| Output | Contents |
|---|---|
| `manuscript_numbers.txt` | every statistic quoted in the text, with the section it belongs to |
| `tables/Table1.csv`, `tables/Table2.csv` | main-text tables (Table 2 now with 95 % confidence intervals) |
| `tables/TableS1…S7.csv` | Supplementary tables, including **S4 exact hyperparameters and seeds**, **S6 sensitivity analysis** and **S7 bootstrap intervals** |
| `figures/fig3…fig6.png` | Figures 3–6 at 600 dpi, redrawn from the final numbers |

New analyses for editor comment 8:

* **Sensitivity analysis** — the main panel-level results recomputed after excluding targets
  with fewer than 10 (and fewer than 20) activity-cliff compounds in the test set.
* **Bootstrap confidence intervals** — for each target, test compounds are resampled with
  replacement (cliff and non-cliff compounds separately, so the cliff count is fixed) 2,000 times.
  Intervals are given for global RMSE, cliff RMSE and the relative cliff penalty Δ. For selection
  optimism, the test-set choice of protocol A is **re-made inside every bootstrap sample**, so the
  interval includes the selection step itself. Intervals are not reported for targets with fewer
  than 5 cliff test compounds, where resampling cannot estimate uncertainty.

In [ ]:
import json
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path

# The notebooks live in the repository folder MTAC_Activity_Cliffs/ and the data in
# MTAC_Activity_Cliffs/Data_analysis/. If you run them from inside Data_analysis/
# instead, the "." fallback finds the same files.
BASE = Path("Data_analysis") if Path("Data_analysis").is_dir() else Path(".")
RAW_DIR = BASE                                  # *_raw.csv, *_filtered.csv, *_curated.csv, *_final.csv
PREP_DIR = BASE / "Data"                        # *_prepared.csv (MoleculeACE input)
PROCESS_DIR = BASE / "Process_Steps"
RESULTS_DIR = BASE / "Results" / "rerun"        # output of 04_rerun_with_validation; every number in the paper comes from here
BACKUP_DIR = BASE / "previous_version"          # copies of files as they were before this rerun
for _d in (PREP_DIR, PROCESS_DIR, RESULTS_DIR):
    _d.mkdir(parents=True, exist_ok=True)
print("Working folder:", BASE.resolve())

In [ ]:
# The 33 human targets of the paper. ChEMBL targets are named by ChEMBL ID; the three
# BindingDB targets by the file stem used for their curated and prepared files.
CHEMBL_TARGETS = [
    "CHEMBL203", "CHEMBL204", "CHEMBL214", "CHEMBL217", "CHEMBL233", "CHEMBL261",
    "CHEMBL262", "CHEMBL267", "CHEMBL279", "CHEMBL299", "CHEMBL301", "CHEMBL331",
    "CHEMBL1827", "CHEMBL1862", "CHEMBL1907601", "CHEMBL1936", "CHEMBL1978", "CHEMBL2276",
    "CHEMBL2695", "CHEMBL2842", "CHEMBL3371", "CHEMBL3587", "CHEMBL4040", "CHEMBL4072",
    "CHEMBL4282", "CHEMBL4482", "CHEMBL4501", "CHEMBL4792", "CHEMBL4816", "CHEMBL5023",
]
BINDINGDB_TARGETS = ["5-HT7_receptor", "Acetylcholinesterase", "Bcl-2"]
ALL_TARGETS = CHEMBL_TARGETS + BINDINGDB_TARGETS
assert len(CHEMBL_TARGETS) == 30 and len(set(ALL_TARGETS)) == 33


TABLE_DIR = RESULTS_DIR / "tables"
FIG_DIR = RESULTS_DIR / "figures"
TABLE_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

N_BOOT = 2000
BOOT_SEED = 42
MIN_CLIFF_FOR_CI = 5
SENSITIVITY_THRESHOLDS = [10, 20]

# name, protein class, source  (row order = order of Tables 1 and 2)
TARGET_INFO = {
    "CHEMBL233":      ("Mu-opioid receptor",   "GPCR (peptidergic)",          "ChEMBL"),
    "CHEMBL214":      ("5-HT1A receptor",      "GPCR (aminergic)",            "ChEMBL"),
    "CHEMBL3371":     ("5-HT6 receptor",       "GPCR (aminergic)",            "ChEMBL"),
    "5-HT7_receptor": ("5-HT7 receptor",       "GPCR (aminergic)",            "BindingDB"),
    "CHEMBL1862":     ("ABL1",                 "Tyrosine kinase",             "ChEMBL"),
    "Acetylcholinesterase": ("Acetylcholinesterase", "Serine hydrolase",      "BindingDB"),
    "CHEMBL4282":     ("AKT1",                 "Ser/Thr kinase",              "ChEMBL"),
    "CHEMBL4816":     ("AKT3",                 "Ser/Thr kinase",              "ChEMBL"),
    "CHEMBL1978":     ("Aromatase",            "Cytochrome P450",             "ChEMBL"),
    "Bcl-2":          ("Bcl-2",                "Protein-protein interaction", "BindingDB"),
    "CHEMBL267":      ("c-Src",                "Tyrosine kinase",             "ChEMBL"),
    "CHEMBL261":      ("Carbonic anhydrase 1", "Lyase (metalloenzyme)",       "ChEMBL"),
    "CHEMBL4072":     ("Cathepsin B",          "Cysteine protease",           "ChEMBL"),
    "CHEMBL301":      ("CDK2",                 "Ser/Thr kinase",              "ChEMBL"),
    "CHEMBL331":      ("CDK4",                 "Ser/Thr kinase",              "ChEMBL"),
    "CHEMBL1907601":  ("CDK4/cyclin D1",       "Ser/Thr kinase complex",      "ChEMBL"),
    "CHEMBL217":      ("D2 dopamine receptor", "GPCR (aminergic)",            "ChEMBL"),
    "CHEMBL203":      ("EGFR",                 "Tyrosine kinase",             "ChEMBL"),
    "CHEMBL4040":     ("ERK2",                 "Ser/Thr kinase",              "ChEMBL"),
    "CHEMBL2695":     ("FAK1",                 "Tyrosine kinase",             "ChEMBL"),
    "CHEMBL262":      ("GSK-3β",               "Ser/Thr kinase",              "ChEMBL"),
    "CHEMBL2276":     ("JNK1",                 "Ser/Thr kinase",              "ChEMBL"),
    "CHEMBL1936":     ("KIT",                  "Tyrosine kinase",             "ChEMBL"),
    "CHEMBL5023":     ("MDM2",                 "Protein-protein interaction", "ChEMBL"),
    "CHEMBL3587":     ("MEK1",                 "Dual-specificity kinase",     "ChEMBL"),
    "CHEMBL2842":     ("mTOR",                 "Ser/Thr kinase",              "ChEMBL"),
    "CHEMBL4792":     ("Orexin receptor 2",    "GPCR (peptidergic)",          "ChEMBL"),
    "CHEMBL4482":     ("PAK4",                 "Ser/Thr kinase",              "ChEMBL"),
    "CHEMBL1827":     ("PDE5",                 "Phosphodiesterase",           "ChEMBL"),
    "CHEMBL299":      ("PKCα",                 "Ser/Thr kinase",              "ChEMBL"),
    "CHEMBL4501":     ("S6K1",                 "Ser/Thr kinase",              "ChEMBL"),
    "CHEMBL204":      ("Thrombin",             "Serine protease",             "ChEMBL"),
    "CHEMBL279":      ("VEGFR2",               "Tyrosine kinase",             "ChEMBL"),
}
assert set(TARGET_INFO) == set(ALL_TARGETS)
ORDER = list(TARGET_INFO)

def name(t):  return TARGET_INFO[t][0]
def tid(t):   return t if t.startswith("CHEMBL") else "BindingDB"

# Published %Cliff in the MoleculeACE benchmark (van Tilborg et al. 2022, Table 1)
MOLECULEACE_PUBLISHED = {"CHEMBL262": 18, "CHEMBL1862": 32, "CHEMBL233": 35,
                         "CHEMBL214": 35, "CHEMBL204": 36, "CHEMBL4792": 52}

## Load the final run

In [ ]:
# Your files may use other names for the three BindingDB targets; map them to one name each
ALIASES = {"5-HT7": "5-HT7_receptor", "5-hydroxytryptamine_receptor_7": "5-HT7_receptor",
           "AChE": "Acetylcholinesterase", "Apoptosis_regulator_Bcl-2": "Bcl-2"}
canon = lambda t: ALIASES.get(str(t), str(t))

res = pd.read_csv(RESULTS_DIR / "per_target_results.csv")
res["target"] = res["target"].map(canon)
missing = sorted(set(ORDER) - set(res.target))
assert not missing, f"targets missing from per_target_results.csv: {missing}"
res = res.set_index("target").loc[ORDER].reset_index()
grid = pd.read_csv(RESULTS_DIR / "full_grid.csv")
grid["target"] = grid["target"].map(canon)
preds = pd.read_csv(RESULTS_DIR / "test_predictions.csv")
preds["target"] = preds["target"].map(canon)

PREPARED = {canon(f.stem.replace("_prepared", "")): f for f in PREP_DIR.glob("*_prepared.csv")}
run_info = json.loads((RESULTS_DIR / "run_info.json").read_text())

# Dataset statistics straight from the prepared files
rows = []
for t in ORDER:
    p = pd.read_csv(PREPARED[t])
    test = p.split == "test"
    rows.append({"target": t, "n": len(p), "n_train": int((~test).sum()), "n_test": int(test.sum()),
                 "pct_cliff": 100 * p.cliff_mol.mean(), "pct_cliff_test": 100 * p.loc[test, "cliff_mol"].mean(),
                 "n_cliff_test": int(p.loc[test, "cliff_mol"].sum())})
ds = pd.DataFrame(rows)

assert len(res) == 33 and (res.n_test.values == ds.n_test.values).all(), "results and prepared files disagree"
res["B_delta"] = res.B_test_cliff_rmse - res.B_test_rmse
res["B_delta_pct"] = 100 * res.B_delta / res.B_test_rmse
res["A_delta_pct"] = 100 * (res.A_test_cliff_rmse - res.A_test_rmse) / res.A_test_rmse

NUM = []          # (section, text) collected for manuscript_numbers.txt
def say(section, text):
    NUM.append((section, text))
    print(f"[{section}] {text}")

def p_fmt(p):
    return f"{p:.1e}" if p < 0.001 else f"{p:.3f}"

## Sections 2.1–2.4 · Panel, cliff frequency and dataset size

In [ ]:
src = pd.Series({t: TARGET_INFO[t][2] for t in ORDER})
say("2.1", f"{len(ORDER)} targets: {int((src == 'ChEMBL').sum())} from ChEMBL, {int((src == 'BindingDB').sum())} from BindingDB")
say("2.1", f"total curated compounds across the panel (sum of n): {ds.n.sum():,}")
classes = pd.Series({t: TARGET_INFO[t][1] for t in ORDER})
say("2.1", f"protein kinases: {int(classes.str.contains('kinase').sum())}; GPCRs: {int(classes.str.startswith('GPCR').sum())}; "
           f"PPI: {int((classes == 'Protein-protein interaction').sum())}")
smallest, largest = ds.loc[ds.n.idxmin()], ds.loc[ds.n.idxmax()]
say("2.1", f"dataset size {smallest.n} ({name(smallest.target)}) to {largest.n:,} ({name(largest.target)})")

pc = ds.set_index("target").pct_cliff
lo, hi = pc.idxmin(), pc.idxmax()
say("2.3", f"%Cliff {pc.min():.2f}% ({name(lo)}) to {pc.max():.2f}% ({name(hi)}); ratio {pc.max() / pc.min():.1f}-fold")
say("2.3", f"%Cliff mean {pc.mean():.1f}%, median {pc.median():.1f}%, IQR {pc.quantile(.25):.1f}% to {pc.quantile(.75):.1f}%")
for lab, sel in [(">50%", pc > 50), ("40-50%", (pc >= 40) & (pc <= 50)), ("<10%", pc < 10)]:
    say("2.3", f"targets {lab}: " + ", ".join(f"{name(t)} ({pc[t]:.2f}%)" for t in pc[sel].sort_values(ascending=False).index))

r, p = stats.pearsonr(ds.n, ds.pct_cliff)
rs, ps = stats.spearmanr(ds.n, ds.pct_cliff)
say("2.4", f"size vs %Cliff: Pearson r = {r:.2f} (p = {p_fmt(p)}), r^2 = {r * r:.2f}; Spearman rho = {rs:.2f} (p = {p_fmt(ps)})")
frac = ds.n_test / ds.n
say("2.4", f"realised test fraction {100 * frac.min():.1f}% to {100 * frac.max():.1f}% (max: {name(ds.loc[frac.idxmax(), 'target'])})")
dev = (ds.pct_cliff - ds.pct_cliff_test).abs()
say("2.4", f"|%Cliff - %Cliff(test)| <= 1 point for {int((dev <= 1).sum())} of 33 targets; max {dev.max():.2f} ({name(ds.loc[dev.idxmax(), 'target'])})")

## Sections 2.5–2.7 · Cross-validated selection (protocol B)

In [ ]:
dc, mc = res.B_descriptor.value_counts(), res.B_model.value_counts()
say("2.5", "descriptor chosen: " + ", ".join(f"{k} {v} ({100 * v / 33:.0f}%)" for k, v in dc.items()))
ecfp = res.B_descriptor == "ECFP"
say("2.5", f"mean cliff RMSE: ECFP-selected {res.loc[ecfp, 'B_test_cliff_rmse'].mean():.3f} vs other {res.loc[~ecfp, 'B_test_cliff_rmse'].mean():.3f}")
say("2.5", "non-ECFP targets: " + "; ".join(f"{name(r.target)} {r.B_descriptor}-{r.B_model} ({r.B_test_rmse:.3f}, {r.B_test_cliff_rmse:.3f})"
                                          for r in res[~ecfp].itertuples()))
say("2.6", "algorithm chosen: " + ", ".join(f"{k} {v} ({100 * v / 33:.0f}%)" for k, v in mc.items()))
say("2.6", f"ECFP+SVM: {int(((res.B_descriptor == 'ECFP') & (res.B_model == 'SVM')).sum())} targets")
low3 = res.nsmallest(3, "B_test_cliff_rmse")
say("2.6", "three lowest cliff RMSE: " + ", ".join(f"{name(r.target)} {r.B_test_cliff_rmse:.3f} ({r.B_descriptor}-{r.B_model})" for r in low3.itertuples()))

worse = res.B_delta > 0
w = stats.wilcoxon(res.B_test_cliff_rmse, res.B_test_rmse)
say("2.7", f"cliff RMSE > global RMSE for {int(worse.sum())} of 33 targets")
say("2.7", f"mean difference {res.B_delta.mean():.3f} pKi; mean increase {res.B_delta_pct.mean():.1f}%, median {res.B_delta_pct.median():.1f}%")
say("2.7", f"Wilcoxon signed-rank W = {w.statistic:.1f}, p = {p_fmt(w.pvalue)}")
say("2.7", f"global RMSE {res.B_test_rmse.min():.3f}-{res.B_test_rmse.max():.3f}; cliff RMSE {res.B_test_cliff_rmse.min():.3f}-{res.B_test_cliff_rmse.max():.3f}")
say("2.7", f"Δ range {res.B_delta_pct.min():+.1f}% to {res.B_delta_pct.max():+.1f}%")
top = res.nlargest(6, "B_delta_pct")
say("2.7", "largest penalties: " + "; ".join(f"{name(r.target)} {r.B_delta_pct:+.1f}% ({r.B_test_rmse:.3f} vs {r.B_test_cliff_rmse:.3f}, n cliff {r.n_cliff_test})" for r in top.itertuples()))
neg = res[res.B_delta <= 0].sort_values("B_delta_pct")
say("2.7", "cliff RMSE <= global: " + "; ".join(f"{name(r.target)} {r.B_delta_pct:+.1f}% (n cliff {r.n_cliff_test})" for r in neg.itertuples()))

## Section 2.8 · Test-set selection versus cross-validated selection

In [ ]:
opt = res.optimism_cliff_rmse
say("2.8", f"same winning combination: {int(res.same_combination.sum())} of 33 ({100 * res.same_combination.mean():.0f}%)")
say("2.8", f"mean cliff RMSE: test-selected {res.A_test_cliff_rmse.mean():.3f}, cv-selected {res.B_test_cliff_rmse.mean():.3f}")
say("2.8", f"selection optimism mean {opt.mean():.3f}, median {opt.median():.3f}, max {opt.max():.3f} ({name(res.loc[opt.idxmax(), 'target'])}); min {opt.min():.3f}")
say("2.8", f"targets with cliff RMSE < global: {int((res.A_delta_pct < 0).sum())} under test selection vs {int((res.B_delta_pct < 0).sum())} under CV selection "
           f"(<= global: {int((res.A_delta_pct <= 0).sum())} vs {int((res.B_delta_pct <= 0).sum())})")
say("2.8", f"largest apparent reduction: {res.A_delta_pct.min():.1f}% (A) vs {res.B_delta_pct.min():.1f}% (B)")
top_a = res.nsmallest(3, "A_delta_pct")
say("2.8", "strongest apparent successes under A: " + "; ".join(
    f"{name(r.target)} {r.A_test_cliff_rmse:.3f} ({r.A_delta_pct:.0f}%) -> {r.B_test_cliff_rmse:.3f} under B" for r in top_a.itertuples()))
r, p = stats.pearsonr(opt, res.n_cliff_test)
rs, ps = stats.spearmanr(opt, res.n_cliff_test)
say("2.8", f"optimism vs n cliff (test): Pearson r = {r:.2f} (p = {p_fmt(p)}); Spearman rho = {rs:.2f} (p = {p_fmt(ps)})")
small = res.n_cliff_test < 20
u = stats.mannwhitneyu(opt[small], opt[~small], alternative="two-sided")
say("2.8", f"optimism: {opt[small].mean():.3f} for {int(small.sum())} targets with <20 cliff test cpds vs {opt[~small].mean():.3f} "
           f"for {int((~small).sum())} with >=20 (Mann-Whitney U = {u.statistic:.0f}, p = {p_fmt(u.pvalue)})")

## Section 2.9 · Comparison with the published MoleculeACE benchmark

In [ ]:
cmp_rows = []
for t, pub in MOLECULEACE_PUBLISHED.items():
    d = ds.set_index("target").loc[t]
    cmp_rows.append({"Protein name": name(t), "Target ID": t, "n (this work)": int(d.n),
                     "%Cliff (this work)": round(d.pct_cliff, 2), "%Cliff (published)": pub,
                     "Difference": round(d.pct_cliff - pub, 1)})
cmp = pd.DataFrame(cmp_rows)
a, b = cmp["%Cliff (published)"], cmp["%Cliff (this work)"]
r, p = stats.pearsonr(a, b); rs, ps = stats.spearmanr(a, b)
say("2.9", "this work vs published: " + "; ".join(f"{x['Protein name']} {x['%Cliff (this work)']:.2f} vs {x['%Cliff (published)']}" for _, x in cmp.iterrows()))
say("2.9", f"Pearson r = {r:.2f} (p = {p_fmt(p)}); Spearman rho = {rs:.2f} (p = {p_fmt(ps)}); difference {cmp.Difference.min():+.1f} to {cmp.Difference.max():+.1f} points")

## Sensitivity analysis: excluding targets with few activity-cliff compounds (new, Table S6)

In [ ]:
def panel_summary(sub, label):
    w = stats.wilcoxon(sub.B_test_cliff_rmse, sub.B_test_rmse)
    return {
        "Subset": label, "Targets": len(sub),
        "Cliff RMSE > global (n)": int((sub.B_delta > 0).sum()),
        "Mean Δ (%)": round(sub.B_delta_pct.mean(), 1), "Median Δ (%)": round(sub.B_delta_pct.median(), 1),
        "Wilcoxon W": round(float(w.statistic), 1), "Wilcoxon p": float(f"{w.pvalue:.2g}"),
        "Mean selection optimism": round(sub.optimism_cliff_rmse.mean(), 3),
        "Median selection optimism": round(sub.optimism_cliff_rmse.median(), 3),
        "Cliff < global, test-selected (n)": int((sub.A_delta_pct < 0).sum()),
        "Cliff < global, CV-selected (n)": int((sub.B_delta_pct < 0).sum()),
    }

sens = [panel_summary(res, "All targets")]
for k in SENSITIVITY_THRESHOLDS:
    sens.append(panel_summary(res[res.n_cliff_test >= k], f"≥ {k} cliff compounds in test set"))
sens = pd.DataFrame(sens)
display(sens.T)
for _, row in sens.iterrows():
    say("S6", f"{row.Subset}: {row.Targets} targets; cliff>global {row['Cliff RMSE > global (n)']}; mean Δ {row['Mean Δ (%)']}%; "
              f"Wilcoxon p = {row['Wilcoxon p']}; mean optimism {row['Mean selection optimism']}; "
              f"cliff<global A/B {row['Cliff < global, test-selected (n)']}/{row['Cliff < global, CV-selected (n)']}")

## Bootstrap confidence intervals (new, Table S7 and Table 2)

In [ ]:
rng = np.random.default_rng(BOOT_SEED)

def target_arrays(t):
    """Squared errors for every combination on the same ordered test set."""
    p = preds[preds.target == t]
    combos, sq, ref = [], [], None
    for (d, m), g in p.groupby(["descriptor", "model"], sort=False):
        smi = g.smiles.to_numpy()
        if ref is None:
            ref, cliff = smi, g.cliff_mol.to_numpy().astype(bool)
        assert (smi == ref).all(), f"{t}: test compounds not aligned across combinations"
        combos.append((d, m))
        sq.append((g.y_true.to_numpy() - g.y_pred.to_numpy()) ** 2)
    return combos, np.vstack(sq), cliff


boot_rows, panel_delta, panel_opt = [], [], []
per_target_reps = {}
for r in res.itertuples():
    combos, sq, cliff = target_arrays(r.target)
    b_idx = combos.index((r.B_descriptor, r.B_model))
    ci, nc = np.where(cliff)[0], np.where(~cliff)[0]
    reps = np.empty((N_BOOT, 4))
    for i in range(N_BOOT):
        sc = rng.choice(ci, size=len(ci), replace=True)
        sn = rng.choice(nc, size=len(nc), replace=True)
        cliff_rmse_all = np.sqrt(sq[:, sc].mean(axis=1))            # every combination
        g_rmse = np.sqrt(np.concatenate([sq[b_idx, sc], sq[b_idx, sn]]).mean())
        c_rmse = cliff_rmse_all[b_idx]
        reps[i] = (g_rmse, c_rmse, 100 * (c_rmse - g_rmse) / g_rmse, c_rmse - cliff_rmse_all.min())
    per_target_reps[r.target] = reps
    ok = len(ci) >= MIN_CLIFF_FOR_CI
    q = lambda col: np.percentile(reps[:, col], [2.5, 97.5]) if ok else (np.nan, np.nan)
    (g_lo, g_hi), (c_lo, c_hi), (d_lo, d_hi), (o_lo, o_hi) = q(0), q(1), q(2), q(3)
    boot_rows.append({"target": r.target, "n_cliff_test": len(ci),
                      "RMSE": r.B_test_rmse, "RMSE CI low": g_lo, "RMSE CI high": g_hi,
                      "Cliff RMSE": r.B_test_cliff_rmse, "Cliff RMSE CI low": c_lo, "Cliff RMSE CI high": c_hi,
                      "Δ (%)": r.B_delta_pct, "Δ CI low": d_lo, "Δ CI high": d_hi,
                      "Selection optimism": r.optimism_cliff_rmse, "Optimism CI low": o_lo, "Optimism CI high": o_hi})
boot = pd.DataFrame(boot_rows)

# Panel level: each bootstrap replicate resamples every target at once
all_reps = np.stack([per_target_reps[t] for t in ORDER])          # (33, N_BOOT, 4)
mean_delta = all_reps[:, :, 2].mean(axis=0)
mean_opt = all_reps[:, :, 3].mean(axis=0)
n_worse = (all_reps[:, :, 2] > 0).sum(axis=0)

est = boot.dropna(subset=["Δ CI low"])
above = int((est["Δ CI low"] > 0).sum()); below = int((est["Δ CI high"] < 0).sum())
say("S7", f"bootstrap ({N_BOOT} replicates, seed {BOOT_SEED}); per-target intervals for {len(est)} targets with >= {MIN_CLIFF_FOR_CI} cliff test compounds")
say("S7", f"Δ interval entirely above 0 for {above} targets, entirely below 0 for {below}, includes 0 for {len(est) - above - below}")
say("S7", f"panel mean Δ {res.B_delta_pct.mean():.1f}% (95% CI {np.percentile(mean_delta, 2.5):.1f} to {np.percentile(mean_delta, 97.5):.1f}%)")
say("S7", f"targets with cliff RMSE > global: {int((res.B_delta > 0).sum())} (95% CI {np.percentile(n_worse, 2.5):.0f} to {np.percentile(n_worse, 97.5):.0f})")
say("S7", f"panel mean selection optimism (re-selected in every replicate) {np.mean(mean_opt):.3f} "
          f"(95% CI {np.percentile(mean_opt, 2.5):.3f} to {np.percentile(mean_opt, 97.5):.3f})")
display(boot.round(3))

## Tables

In [ ]:
def ci(lo, hi, nd=3):
    return "" if pd.isna(lo) else f"{lo:.{nd}f}–{hi:.{nd}f}"

dsi = ds.set_index("target"); ri = res.set_index("target"); bi = boot.set_index("target")

table1 = pd.DataFrame([{
    "Protein Name": name(t), "Target ID": tid(t), "Protein Class": TARGET_INFO[t][1],
    "n (n test)": f"{dsi.loc[t, 'n']:,} ({dsi.loc[t, 'n_test']:,})",
    "%Cliff (%Cliff test)": f"{dsi.loc[t, 'pct_cliff']:.2f} ({dsi.loc[t, 'pct_cliff_test']:.2f})",
} for t in ORDER])

table2 = pd.DataFrame([{
    "Protein Name": name(t), "Target ID": tid(t),
    "Descriptor": ri.loc[t, "B_descriptor"], "Model": ri.loc[t, "B_model"],
    "RMSE": f"{ri.loc[t, 'B_test_rmse']:.3f}",
    "Cliff RMSE": f"{ri.loc[t, 'B_test_cliff_rmse']:.3f}",
    "Cliff RMSE 95% CI": ci(bi.loc[t, "Cliff RMSE CI low"], bi.loc[t, "Cliff RMSE CI high"]),
    "n cliff": int(ri.loc[t, "n_cliff_test"]),
    "Δ (%)": f"{ri.loc[t, 'B_delta_pct']:+.1f}",
    "Δ 95% CI": ci(bi.loc[t, "Δ CI low"], bi.loc[t, "Δ CI high"], 1),
} for t in ORDER])

tableS1 = pd.DataFrame([{
    "Protein name": name(t), "Target ID": tid(t), "Protein class": TARGET_INFO[t][1],
    "Source": TARGET_INFO[t][2], "n": dsi.loc[t, "n"], "n train": dsi.loc[t, "n_train"], "n test": dsi.loc[t, "n_test"],
    "%Cliff": round(dsi.loc[t, "pct_cliff"], 2), "%Cliff test": round(dsi.loc[t, "pct_cliff_test"], 2),
    "n cliff (test)": dsi.loc[t, "n_cliff_test"],
} for t in ORDER])

tableS2 = table2.assign(_d=[ri.loc[t, "B_delta_pct"] for t in ORDER]).sort_values("_d", ascending=False).drop(columns="_d")
tableS2["Protein Name"] = [n + (" ᵃ" if k < 10 else "") for n, k in zip(tableS2["Protein Name"], tableS2["n cliff"])]

tableS4 = pd.DataFrame([{"Algorithm": k.split("_")[0], "Descriptor": k.split("_")[1],
                         "Hyperparameter": p, "Value": "None" if v is None else str(v)}
                        for k, params in run_info["hyperparameters"].items() for p, v in params.items()])

tableS5 = pd.DataFrame([{
    "Protein name": name(t), "Target ID": tid(t), "n cliff": int(ri.loc[t, "n_cliff_test"]),
    "Test-selected": f"{ri.loc[t, 'A_descriptor']}-{ri.loc[t, 'A_model']}",
    "Cliff RMSE (test-selected)": round(ri.loc[t, "A_test_cliff_rmse"], 3),
    "CV-selected": f"{ri.loc[t, 'B_descriptor']}-{ri.loc[t, 'B_model']}",
    "Cliff RMSE (CV-selected)": round(ri.loc[t, "B_test_cliff_rmse"], 3),
    "Selection optimism": round(ri.loc[t, "optimism_cliff_rmse"], 3),
    "Optimism 95% CI": ci(bi.loc[t, "Optimism CI low"], bi.loc[t, "Optimism CI high"]),
} for t in ORDER])

tableS7 = boot.assign(**{"Protein name": [name(t) for t in boot.target]}).drop(columns="target")

for nm, tb in [("Table1", table1), ("Table2", table2), ("TableS1", tableS1), ("TableS2", tableS2),
               ("TableS3", cmp), ("TableS4", tableS4), ("TableS5", tableS5), ("TableS6", sens), ("TableS7", tableS7)]:
    tb.to_csv(TABLE_DIR / f"{nm}.csv", index=False)
print("Tables written to", TABLE_DIR)
display(table2)

## Figures 3–6 (600 dpi)

In [ ]:
matplotlib.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 8, "axes.edgecolor": "#5a5a5a", "axes.linewidth": 0.6,
    "xtick.color": "#5a5a5a", "ytick.color": "#5a5a5a", "text.color": "#1a1a1a", "axes.labelcolor": "#1a1a1a",
    "figure.facecolor": "white", "axes.facecolor": "white", "savefig.facecolor": "white", "legend.frameon": False})
BLUE, GREEN, ORANGE, PURPLE = "#0072B2", "#009E73", "#D55E00", "#CC79A7"   # Okabe-Ito, colour-blind safe
GRID = "#d9d9d9"
NT = len(res)

def clean(ax, keep_left=True):
    for s in ("top", "right") + (() if keep_left else ("left",)):
        ax.spines[s].set_visible(False)
    ax.set_axisbelow(True)

# Figure 3: descriptors
order = ["ECFP", "MACCS", "PHYSCHEM", "WHIM"]
vals = [int(dc.get(k, 0)) for k in order]
fig, ax = plt.subplots(figsize=(4.6, 2.9))
bars = ax.bar(order, vals, color=[BLUE, PURPLE, ORANGE, GREEN], width=0.62, zorder=3)
for b_, v in zip(bars, vals):
    ax.text(b_.get_x() + b_.get_width() / 2, v + 0.4, f"{v}\n({100 * v / NT:.0f}%)", ha="center", va="bottom", fontsize=8)
ax.set_ylabel(f"Targets for which descriptor was optimal (n = {NT})")
ax.set_ylim(0, max(vals) * 1.3); ax.grid(axis="y", color=GRID, lw=0.5, zorder=0); clean(ax)
fig.tight_layout(); fig.savefig(FIG_DIR / "fig3_descriptors.png", dpi=600); plt.close(fig)

# Figure 4: algorithms
order = ["SVM", "RF", "GBM", "KNN"]
vals = [int(mc.get(k, 0)) for k in order]
fig, ax = plt.subplots(figsize=(4.6, 2.6))
y = np.arange(len(order))[::-1]
bars = ax.barh(y, vals, color=[BLUE, GREEN, ORANGE, PURPLE], height=0.6, zorder=3)
ax.set_yticks(y); ax.set_yticklabels(order); ax.tick_params(axis="y", length=0)
for b_, v in zip(bars, vals):
    ax.text(v + 0.3, b_.get_y() + b_.get_height() / 2, f"{v} targets ({100 * v / NT:.0f}%)", va="center", fontsize=8)
ax.set_xlabel(f"Targets for which algorithm was optimal (n = {NT})")
ax.set_xlim(0, max(vals) * 1.35); ax.grid(axis="x", color=GRID, lw=0.5, zorder=0); clean(ax, keep_left=False)
fig.tight_layout(); fig.savefig(FIG_DIR / "fig4_algorithms.png", dpi=600); plt.close(fig)

# Figure 5: relative cliff penalty per target, with bootstrap intervals
f5 = res.assign(nm=[name(t) for t in res.target]).merge(boot[["target", "Δ CI low", "Δ CI high"]], on="target")
f5 = f5.sort_values("B_delta_pct")
fig, ax = plt.subplots(figsize=(5.4, 6.6))
y = np.arange(len(f5))
ax.barh(y, f5.B_delta_pct, color=[BLUE if v < 0 else ORANGE for v in f5.B_delta_pct], height=0.68, zorder=3)
has = f5["Δ CI low"].notna().to_numpy()
ax.errorbar(f5.B_delta_pct[has], y[has],
            xerr=[f5.B_delta_pct[has] - f5["Δ CI low"][has], f5["Δ CI high"][has] - f5.B_delta_pct[has]],
            fmt="none", ecolor="#1a1a1a", elinewidth=0.6, capsize=1.5, zorder=4)
ax.set_yticks(y)
ax.set_yticklabels([f"{n}{'*' if k < MIN_CLIFF_FOR_CI else ''} ({k})" for n, k in zip(f5.nm, f5.n_cliff_test)], fontsize=7)
ax.tick_params(axis="y", length=0); ax.axvline(0, color="#5a5a5a", lw=0.8, zorder=5)
ax.set_xlabel("Cliff RMSE relative to global RMSE, Δ (%)")
ax.grid(axis="x", color=GRID, lw=0.5, zorder=0); clean(ax, keep_left=False)
fig.text(0.99, 0.005, "Error bars: 95% bootstrap interval. (n) = cliff compounds in the test set; "
         f"* fewer than {MIN_CLIFF_FOR_CI}, interval not estimable.", ha="right", va="bottom", fontsize=6.5, color="#5a5a5a")
fig.tight_layout(rect=(0, 0.025, 1, 1)); fig.savefig(FIG_DIR / "fig5_delta_rmse.png", dpi=600); plt.close(fig)

# Figure 6: (a) test-selected vs CV-selected cliff RMSE, (b) optimism vs number of cliff compounds
fig, (a1, a2) = plt.subplots(1, 2, figsize=(7.2, 3.2))
a1.scatter(res.A_test_cliff_rmse, res.B_test_cliff_rmse, s=14, color=BLUE, zorder=3)
lims = [min(res.A_test_cliff_rmse.min(), res.B_test_cliff_rmse.min()) - 0.05,
        max(res.A_test_cliff_rmse.max(), res.B_test_cliff_rmse.max()) + 0.05]
a1.plot(lims, lims, ls="--", color="#5a5a5a", lw=0.8); a1.set_xlim(lims); a1.set_ylim(lims)
a1.set_xlabel("Cliff RMSE, selected on the test set"); a1.set_ylabel("Cliff RMSE, selected by cross-validation")
a1.text(0.03, 0.97, "(a)", transform=a1.transAxes, va="top", fontweight="bold")
a1.grid(color=GRID, lw=0.5, zorder=0); clean(a1)
a2.scatter(res.n_cliff_test, res.optimism_cliff_rmse, s=14, color=ORANGE, zorder=3)
a2.set_xscale("log"); a2.axhline(0, color="#5a5a5a", lw=0.8)
r6, p6 = stats.pearsonr(res.optimism_cliff_rmse, res.n_cliff_test)
a2.text(0.97, 0.97, f"r = {r6:.2f}, p = {p_fmt(p6)}", transform=a2.transAxes, ha="right", va="top", fontsize=7)
a2.set_xlabel("Activity-cliff compounds in the test set"); a2.set_ylabel("Selection optimism (pKi units)")
a2.text(0.03, 0.97, "(b)", transform=a2.transAxes, va="top", fontweight="bold")
a2.grid(color=GRID, lw=0.5, zorder=0); clean(a2)
fig.tight_layout(); fig.savefig(FIG_DIR / "fig6_selection.png", dpi=600); plt.close(fig)
print("Figures written to", FIG_DIR)

In [ ]:
lines = ["Numbers for the manuscript, computed by notebook 05 from Results/rerun/", ""]
section = None
for sec, txt in NUM:
    if sec != section:
        lines += ["", f"=== Section {sec} ==="]
        section = sec
    lines.append(txt)
(RESULTS_DIR / "manuscript_numbers.txt").write_text("\n".join(lines))
print("Wrote", RESULTS_DIR / "manuscript_numbers.txt")